# Translation of data - NL2EN

In [9]:
%pip install transformers accelerate
%pip install ctranslate2 OpenNMT-py==2.* sentencepiece
%pip install sacremoses

     |████████████████████████████████| 330 kB 11.9 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.
zsh:1: no matches found: OpenNMT-py==2.*
Note: you may need to restart the kernel to use updated packages.
     |████████████████████████████████| 897 kB 10.6 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [10]:
%pip install sentencepiece

     |████████████████████████████████| 1.3 MB 11.2 MB/s eta 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [5]:
#%pip install transformers -U
import transformers

In [1]:
import ctranslate2

#https://opennmt.net/CTranslate2/guides/transformers.html#marianmt
!ct2-transformers-converter --model Helsinki-NLP/opus-mt-nl-en --output_dir /workspace/mijnidbcoachnlp/data/ct2_model --force

model.safetensors: 100%|██████████████████████| 316M/316M [00:01<00:00, 286MB/s]


In [1]:
# Load dataset
import pandas as pd
import tqdm

path = "/workspace/persistent/mijnidbcoachnlp/data/analysis_data/clean_message_data.xlsx"
df = pd.read_excel(path, index_col=0)
df.tail()

,provider_name,program_name,user_guid,date_time,message,direction,sender,message_id,clean_message_placeholder,clean_message
32682,MUMC MijnIBDcoach,MijnIBDcoach5.0,D46C4DD4-89AB-4B3B-B9F6-D96659A0A425,2022-10-18 16:46:37.640,Ontvangen! Dankjewel! Kunnen jullie me vertell...,From client,NaN,32683,Ontvangen! Dankjewel! Kunnen jullie me vertell...,Ontvangen! Dankjewel! Kunnen jullie me vertell...
32683,MUMC MijnIBDcoach,MijnIBDcoach5.0,DE8E6C76-8D3A-4F10-9726-49230F2926CF,2022-10-18 17:53:14.610,"Beste [PERSOON], Ik ben verhuist naar een nieu...",From client,NaN,32684,"[PERSOON], Ik ben verhuist naar een nieuw adre...",Ik ben verhuist naar een nieuw adres: Zou u...
32684,MUMC MijnIBDcoach,MijnIBDcoach5.0,62EF0F66-2D13-480B-B590-6755DBFB2CBC,2022-10-18 19:36:00.757,"Hoi [PERSOON], Zou je mij 2 QuantOn Cal test ...",From client,NaN,32685,"[PERSOON], Zou je mij 2 QuantOn Cal test will...",Zou je mij 2 QuantOn Cal test willen sturen.
32685,MUMC MijnIBDcoach,MijnIBDcoach5.0,06155C22-39C9-4C67-9C30-5DBD8FC4E805,2022-10-19 09:09:34.927,Ik heb een vraag. Via de periodieke vragenlijs...,From client,NaN,32686,Ik heb een vraag. Via de periodieke vragenlijs...,Ik heb een vraag. Via de periodieke vragenlijs...
32686,MUMC MijnIBDcoach,MijnIBDcoach5.0,5E45A065-6BCD-4DC3-96BF-1664F6C70824,2022-10-19 09:19:18.327,"Hallo [PERSOON] Gisteren, laat in de middag, p...",From client,NaN,32687,"[PERSOON] Gisteren, laat in de middag, potje e...","Gisteren, laat in de middag, potje en formulie..."


In [3]:
messages = df["message"].tolist()

In [5]:
import ctranslate2
import transformers
import tqdm

# Load CTranslate2 Translator and the tokenizer
translator = ctranslate2.Translator('/workspace/persistent/mijnidbcoachnlp/data/ct2_model')
tokenizer = transformers.AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-nl-en")

# Function to translate a batch of messages using CTranslate2
def translate_batch(messages, batch_size=1000):
    translations = []
    
    for i in tqdm.tqdm(range(0, len(messages), batch_size)):
        batch = messages[i:i + batch_size]
        
        # Tokenize the batch of messages into tokens (lists of token strings)
        source_batch = [tokenizer.convert_ids_to_tokens(tokenizer.encode(message, add_special_tokens=True)) for message in batch]

        # Translate the batch using CTranslate2 (expects a list of tokenized inputs)
        results = translator.translate_batch(source_batch)

        # Decode the hypotheses (translations) back to text
        for result in results:
            translated_tokens = result.hypotheses[0]  # Access the first hypothesis (most likely translation)
            translated_text = tokenizer.convert_tokens_to_string(translated_tokens)  # Convert tokens back to string
            translations.append(translated_text)
        
        # Print original and translated messages
        #for original, translated in zip(batch, translations[-len(batch):]):
            #print(f"Original: {original}")
            #print(f"Translated: {translated}")
    
    return translations

# Function to clean and filter valid text messages
def clean_messages(messages):
    return [str(message) for message in messages if isinstance(message, str) and message.strip()]

# Clean the messages to remove any non-string or empty values
cleaned_messages = clean_messages(messages)

# Translate messages in batches using CTranslate2
translated_messages = translate_batch(cleaned_messages)

# Add the translations back into the DataFrame (optional)
df["translated_message"] = [None if not isinstance(msg, str) else translated_messages.pop(0) for msg in messages]

# Save the translated messages to a new file (optional)
df.to_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/clean_message_data.xlsx", index=False)


  0%|          | 0/33 [00:00<?, ?it/s]

In [7]:
# read the message df and map the language to the sentences
df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx", index_col=0)
message_df = pd.read_excel("/workspace/persistent/mijnidbcoachnlp/data/analysis_data/message_data.xlsx", index_col=0)

df = df.merge(message_df[['message_id']], on="message_id", how='left')
df

,Message_ID,Sentence,Clean_Sentence,Sentence_ID,Translated_Sentence
0,3,Ik ben 2 weken geleden met spoed opgenomen in ...,Ik ben 2 weken geleden met spoed opgenomen in ...,1,I was rushed into the [PRESSION] two weeks ago...
1,3,"Ik kreeg acuut een pijnlijke druk op de borst,...","Ik kreeg acuut een pijnlijke druk op de borst,...",2,I was acutely under a painful pressure on the ...
2,3,Het begon 1 uur na het avondeten.,Het begon 1 uur na het avondeten.,3,It started 1 hour after dinner.
3,3,"Ik had al de hele dag migraine, had dus ook we...","Ik had al de hele dag migraine, had dus ook we...",4,"I had migraines all day, so I hadn't eaten much."
4,3,"Ik werd heel erg misselijk, braakneigingen, du...","Ik werd heel erg misselijk, braakneigingen, du...",5,"I got very nauseous, vomiting, dizzy and rejuv..."
...,...,...,...,...,...
41138,32679,"Ze hebben de urine op kweek gezet, kan morgenv...","Ze hebben de urine op kweek gezet, kan morgenv...",41139,"They've put the urine on culture, can get the ..."
41139,32679,Afgelopen jaren is er al vaker bloed in mijn u...,Afgelopen jaren is er al vaker bloed in mijn u...,41140,"In recent years, blood has been found in my ur..."
41140,32679,Wat adviseert u hiermee te doen?,Wat adviseert u hiermee te doen?,41141,What do you suggest you do with this?
41141,32681,"Hoi [PERSOON], Oke, dat is iig al een gerustst...","Oke, dat is iig al een geruststelling ik wach...",41142,"Hi [PRESSON], Okay, that's a relief;) I'll wai..."


In [8]:
df.to_excel("/workspace/mijnidbcoachnlp/data/analysis_data/translated_sentence_data_new.xlsx", index=False)
